# ZeroDCE Low-Light Enhancement Kaggle Notebook
This notebook demonstrates how to deploy and test the ZeroDCE model from the GitHub repository [Zero-DCE-Lowlight-Enhancement](https://github.com/Shampavi-Premananthan/Zero-DCE-Lowlight-Enhancement.git) for low-light image enhancement.

In [ ]:
# Clone the ZeroDCE Repository
!git clone https://github.com/Shampavi-Premananthan/Zero-DCE-Lowlight-Enhancement.git

In [ ]:
# Install Required Dependencies
!pip install -q torch torchvision numpy pillow scikit-image
!pip install -q -r Zero-DCE-Lowlight-Enhancement/requirements.txt

In [ ]:
# Import Libraries
import torch
import numpy as np
from PIL import Image
import os
import sys
sys.path.append('Zero-DCE-Lowlight-Enhancement')

In [ ]:
# Load and Preprocess Low-Light Images
# Example: Load a sample image from the repo or upload your own
sample_img_path = 'Zero-DCE-Lowlight-Enhancement/data/test_data/low/1.png'
img = Image.open(sample_img_path).convert('RGB')
img = img.resize((512, 512))
img_np = np.array(img).astype(np.float32) / 255.0
img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0)

In [ ]:
# Load the ZeroDCE Model
from Zero_DCE import ZeroDCE
model = ZeroDCE()
checkpoint_path = 'Zero-DCE-Lowlight-Enhancement/snapshots/Epoch99.pth'
model.load_state_dict(torch.load(checkpoint_path, map_location='cpu'))
model.eval()

In [ ]:
# Enhance Images Using ZeroDCE
with torch.no_grad():
    enhanced_tensor = model(img_tensor)
enhanced_np = enhanced_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
enhanced_img = Image.fromarray((enhanced_np * 255).astype(np.uint8))

In [ ]:
# Visualize Enhanced Results
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(img)
axs[0].set_title('Original Low-Light')
axs[0].axis('off')
axs[1].imshow(enhanced_img)
axs[1].set_title('Enhanced by ZeroDCE')
axs[1].axis('off')
plt.show()

In [ ]:
# Evaluate Enhancement Quality
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
original_np = np.array(img)
enhanced_np_eval = np.array(enhanced_img.resize(img.size))
psnr = peak_signal_noise_ratio(original_np, enhanced_np_eval, data_range=255)
ssim = structural_similarity(original_np, enhanced_np_eval, channel_axis=2, data_range=255)
print(f'PSNR: {psnr:.2f}, SSIM: {ssim:.4f}')

In [ ]:
# Install FastAPI, Uvicorn, pyngrok, and other dependencies for deployment
!pip install -q fastapi uvicorn python-multipart pyngrok pillow timm einops scikit-image

In [ ]:
# FastAPI app and inference helpers for ZeroDCE
import io, uuid, time, base64
import numpy as np
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
import uvicorn
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from typing import Optional
import torch
from Zero_DCE import ZeroDCE
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ZeroDCE()
checkpoint_path = 'Zero-DCE-Lowlight-Enhancement/snapshots/Epoch99.pth'
model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()
def pil_to_tensor(img):
    arr = np.array(img).astype(np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
def tensor_to_pil(t):
    arr = t.squeeze(0).permute(1, 2, 0).cpu().numpy()
    arr = np.clip(arr * 255, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)
def enhance_image_bytes(img_bytes: bytes) -> tuple[bytes, float, float, float]:
    start = time.time()
    orig = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    x = pil_to_tensor(orig).to(DEVICE)
    with torch.no_grad():
        out = model(x)
    enhanced = tensor_to_pil(out)
    orig_arr = np.array(orig)
    enhanced_arr = np.array(enhanced.resize(orig.size))
    p = round(float(psnr_fn(orig_arr, enhanced_arr, data_range=255)), 2)
    s = round(float(ssim_fn(orig_arr, enhanced_arr, channel_axis=2, data_range=255)), 4)
    buf = io.BytesIO()
    enhanced.save(buf, format='PNG')
    elapsed = round(time.time() - start, 2)
    return buf.getvalue(), p, s, elapsed
app = FastAPI(title='ZeroDCE API')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])
SESSIONS = {}
class QualityRequest(BaseModel):
    session_id: str
@app.get('/')
def root():
    return {'status': 'ok', 'gpu': DEVICE}
@app.get('/health')
def health():
    return {'status': 'ok', 'gpu': DEVICE, 'cuda_name': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'N/A'}
@app.post('/api/enhance/fast')
async def enhance_fast(image: UploadFile = File(...)):
    allowed = {'image/jpeg', 'image/jpg', 'image/png'}
    if image.content_type not in allowed:
        raise HTTPException(400, 'Only JPG / PNG allowed.')
    contents = await image.read()
    if len(contents) > 10 * 1024 * 1024:
        raise HTTPException(400, 'File too large (max 10 MB).')
    session_id = str(uuid.uuid4())
    SESSIONS[session_id] = contents
    try:
        enhanced_bytes, p, s, elapsed = enhance_image_bytes(contents)
    except Exception as e:
        raise HTTPException(500, str(e))
    orig_b64 = base64.b64encode(contents).decode()
    enhanced_b64 = base64.b64encode(enhanced_bytes).decode()
    return {'success': True, 'session_id': session_id, 'original_url': f'data:image/png;base64,{orig_b64}', 'image_url': f'data:image/png;base64,{enhanced_b64}', 'psnr': p, 'ssim': s, 'mode': 'fast', 'processing_time': elapsed}
@app.post('/api/enhance/quality')
async def enhance_quality(req: QualityRequest):
    contents = SESSIONS.get(req.session_id)
    if contents is None:
        raise HTTPException(404, 'Session not found. Please upload image again.')
    try:
        enhanced_bytes, p, s, elapsed = enhance_image_bytes(contents)
    except Exception as e:
        raise HTTPException(500, str(e))
    enhanced_b64 = base64.b64encode(enhanced_bytes).decode()
    return {'success': True, 'session_id': req.session_id, 'image_url': f'data:image/png;base64,{enhanced_b64}', 'psnr': p, 'ssim': s, 'mode': 'quality', 'processing_time': elapsed}

In [ ]:
# Start ngrok tunnel and run FastAPI server (foreground, live logs)
import nest_asyncio
from pyngrok import ngrok
nest_asyncio.apply()
NGROK_AUTH_TOKEN = 'PASTE_YOUR_NGROK_AUTH_TOKEN_HERE'  # Replace with your ngrok token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()
public_url = ngrok.connect(8000)
print('=' * 60)
print(f'  🚀 Backend URL: {public_url}')
print('  Copy this URL into .env.local as NEXT_PUBLIC_BACKEND_URL')
print('=' * 60)
print('  Uvicorn starting — live request logs will appear below:')
print('  (Stop this cell to shut down the server)')
print('=' * 60)
config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='info')
server = uvicorn.Server(config)
await server.serve()